In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


In [4]:
df = pd.read_csv('./data/annotated_filtered_col.CG_2.fast.tsv', sep='\t')      # or read_parquet / feather …
df

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,101,200,0.8951,350,41,6,False,False,False,False,0.8951
1,0,1,301,400,0.5487,62,51,2,False,False,False,False,0.5487
2,0,1,401,500,0.8246,47,10,1,False,False,False,False,0.8246
3,0,1,501,600,0.7206,98,38,3,False,False,False,False,0.7206
4,0,1,601,700,0.8982,203,23,6,False,False,False,False,0.8982
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5925984,16,5,26974801,26974900,0.8462,22,4,7,False,False,False,False,0.8462
5925985,16,5,26974901,26975000,0.7500,3,1,2,False,False,False,False,NaN
5925986,16,5,26975101,26975200,1.0000,4,0,4,False,False,False,False,NaN
5925987,16,5,26975201,26975300,1.0000,8,0,8,False,False,False,False,1.0000


In [5]:
import numpy as np, pandas as pd
from scipy.special import logit, expit
from scipy.stats import chi2

# ---------- helpers (unchanged) ----------
def drop_uniformly_low(df):
    # Keep windows where ANY cluster survived the masking
    g = df.groupby(['chr','start','end'])['score_masked'].apply(lambda s: s.notna().any())
    keep = g[g].reset_index()[['chr','start','end']]
    return df.merge(keep, on=['chr','start','end'], how='inner')

def fit_offsets(df_all):
    # aggregate across ALL windows (no categories)
    agg = df_all.groupby('cluster')[['c','t']].sum()
    M = agg.sum(axis=1)
    p = agg['c'] / M
    d = logit(np.clip(p, 1e-6, 1-1e-6))
    # center by M-weighted mean
    return d - np.average(d, weights=M)

def window_stats(df_all, deltas, tau=20.0, eps=1e-6):
    out = []
    # Empirical-Bayes prior from ALL windows
    agg = df_all.groupby('cluster')[['c','t']].sum()
    mu = agg['c'].sum() / agg.sum(axis=1).sum()
    a0, b0 = mu * tau, (1 - mu) * tau

    for (chr_, start, end), g in df_all.groupby(['chr','start','end']):
        c = g['c'].to_numpy(); t = g['t'].to_numpy(); m = c + t
        k = g['cluster'].to_numpy()
        keep = m >= 5 #window omitted from per-cluster x2 calculation if cov below 5
        if keep.sum() < 2: #essentially mean no coverage across any cluster
            continue
        c, m, k = c[keep], m[keep], k[keep]

        # Null with global cluster offsets
        pbar = c.sum() / m.sum()
        p0 = expit(logit(np.clip(pbar, 1e-6, 1-1e-6)) + deltas.loc[k].to_numpy())

        # Pearson X^2 with binomial variance, epsilon-stabilized
        E = m * p0
        Var = m * p0 * (1 - p0) + eps
        X2 = ((c - E)**2 / Var).sum()
        dfree = len(c) - 1

        # EB-shrunk effect spread and argmax/argmin clusters
        p_tilde = (c + a0) / (m + a0 + b0)
        dmax = float(p_tilde.max() - p_tilde.min())
        hi = int(k[p_tilde.argmax()])
        lo = int(k[p_tilde.argmin()])

        out.append((chr_, start, end, X2, dfree, dmax, hi, lo))

    res = pd.DataFrame(out, columns=['chr','start','end','X2','df','delta_max','hi_cluster','lo_cluster'])

    # Global overdispersion phi (from ALL windows)
    phi = np.median(res['X2'] / np.maximum(res['df'], 1)) if len(res) else 1.0
    res['pval'] = 1 - chi2.cdf(res['X2'] / max(phi, 1e-6), res['df'])
    res['phi'] = phi
    return res

def bh_fdr(p):
    if len(p) == 0:
        return p
    r = np.argsort(p)
    ranks = np.empty_like(r); ranks[r] = np.arange(1, len(p) + 1)
    q = p * len(p) / np.maximum(ranks, 1)
    q_sorted = np.minimum.accumulate(np.sort(q)[::-1])[::-1]
    out = np.empty_like(q_sorted); out[r] = q_sorted
    return np.clip(out, 0, 1)



In [6]:

# ---------- run (no categories) ----------
# Start from your original df (with columns: cluster, chr, start, end, c, t, score_masked, ...flags...)
df1 = drop_uniformly_low(df)      # same mask rule; flags are ignored
deltas = fit_offsets(df1)         # global cluster offsets over ALL windows
stats = window_stats(df1, deltas, tau=20.0)
stats['qval'] = bh_fdr(stats['pval'].to_numpy())
stats
# Final DMW table across ALL windows (no category column)
#dmw = stats.sort_values('qval').reset_index(drop=True)


,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,pval,phi,qval
0,1,101,200,19.659218,16,0.146663,3,15,8.002244e-01,1.763387,1.000000
1,1,301,400,25.116873,12,0.301004,14,4,2.854276e-01,1.763387,0.733817
2,1,401,500,51.512517,10,0.349390,0,4,1.151156e-03,1.763387,0.016226
3,1,501,600,95.124356,13,0.411961,11,6,6.185705e-07,1.763387,0.000028
4,1,601,700,53.648165,13,0.221280,3,13,4.090721e-03,1.763387,0.043505
...,...,...,...,...,...,...,...,...,...,...,...
353843,5,26974801,26974900,27.897148,14,0.182474,5,11,3.244772e-01,1.763387,0.780416
353844,5,26974901,26975000,23.686848,11,0.258215,3,12,2.659942e-01,1.763387,0.708799
353845,5,26975101,26975200,8.161012,12,0.131118,1,11,9.692644e-01,1.763387,1.000000
353846,5,26975201,26975300,28.889855,16,0.198657,3,12,4.265540e-01,1.763387,0.879449


In [9]:
stats

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,pval,phi,qval
0,1,101,200,19.659218,16,0.146663,3,15,8.002244e-01,1.763387,1.000000
1,1,301,400,25.116873,12,0.301004,14,4,2.854276e-01,1.763387,0.733817
2,1,401,500,51.512517,10,0.349390,0,4,1.151156e-03,1.763387,0.016226
3,1,501,600,95.124356,13,0.411961,11,6,6.185705e-07,1.763387,0.000028
4,1,601,700,53.648165,13,0.221280,3,13,4.090721e-03,1.763387,0.043505
...,...,...,...,...,...,...,...,...,...,...,...
353843,5,26974801,26974900,27.897148,14,0.182474,5,11,3.244772e-01,1.763387,0.780416
353844,5,26974901,26975000,23.686848,11,0.258215,3,12,2.659942e-01,1.763387,0.708799
353845,5,26975101,26975200,8.161012,12,0.131118,1,11,9.692644e-01,1.763387,1.000000
353846,5,26975201,26975300,28.889855,16,0.198657,3,12,4.265540e-01,1.763387,0.879449


In [7]:
stats.to_csv("./data/x2_stat_all_new.csv")

In [8]:
stats.to_pickle("./data/x2_stat_all_new.pkl")

In [10]:
dmw =  stats[(stats['qval']<0.01) & (stats['delta_max']>0.15)]
dmw

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,pval,phi,qval
3,1,501,600,95.124356,13,0.411961,11,6,6.185705e-07,1.763387,0.000028
8,1,2001,2100,67.937170,13,0.498788,16,0,2.378642e-04,1.763387,0.004551
10,1,6501,6600,90.298156,16,0.470616,16,1,1.469628e-05,1.763387,0.000439
12,1,10501,10600,78.175883,15,0.437942,15,0,9.751773e-05,1.763387,0.002186
13,1,11201,11300,75.572792,14,0.456539,12,0,9.031319e-05,1.763387,0.002050
...,...,...,...,...,...,...,...,...,...,...,...
353822,5,26961101,26961200,204.238737,15,0.492103,16,1,0.000000e+00,1.763387,0.000000
353823,5,26961501,26961600,109.144782,14,0.397873,0,6,5.443627e-08,1.763387,0.000003
353825,5,26961801,26961900,97.995155,16,0.312906,0,11,2.863121e-06,1.763387,0.000107
353833,5,26970801,26970900,239.392816,16,0.340932,5,10,0.000000e+00,1.763387,0.000000


In [16]:
# dmw = stats[(stats['qval']<0.01) & (stats['delta_max']>0.3)]
# dmw #18000row

In [11]:
def merge_windows_with_stats(df, max_gap=150, min_len=100):
    x = df.copy()
    x['chr'] = x['chr'].astype(str)
    x = x.sort_values(['chr','start','end']).reset_index(drop=True)
    x['prev_end'] = x.groupby('chr')['end'].shift()
    x['gap'] = x['start'] - x['prev_end'] - 1
    x['new_block'] = (x['gap'].isna()) | (x['gap'] >= max_gap)
    x['block_id'] = x.groupby('chr')['new_block'].cumsum()

    merged = (
        x.groupby(['chr','block_id'], as_index=False)
         .agg(start=('start','min'),
              end=('end','max'),
              n_windows=('start','size'),
              length_bp=('end', lambda s: s.max())  # temp; fix below
         )
    )
    # recompute length properly
    merged['length_bp'] = merged['end'] - merged['start'] + 1
    # Join back for stats
    stats = (
        x.groupby(['chr','block_id'])
         .agg(min_qval=('qval','min'),
              max_delta=('delta_max','max'),
              mean_delta=('delta_max','mean'))
         .reset_index()
    )
    merged = merged.drop(columns='length_bp').merge(stats, on=['chr','block_id'])
    merged['length_bp'] = merged['end'] - merged['start'] + 1
    merged = merged[merged['length_bp'] > min_len]
    return merged[['chr','start','end','length_bp','n_windows','min_qval','max_delta','mean_delta']].sort_values(['chr','start']).reset_index(drop=True)


In [13]:
chunks = merge_windows_with_stats(dmw, max_gap=150, min_len=100)
chunks

,chr,start,end,length_bp,n_windows,min_qval,max_delta,mean_delta
0,1,76801,77300,500,3,0.000000e+00,0.587940,0.556828
1,1,116901,117300,400,3,0.000000e+00,0.537025,0.479695
2,1,281801,282200,400,3,1.226815e-03,0.270856,0.247911
3,1,282601,282800,200,2,4.359180e-08,0.400563,0.383799
4,1,292201,292600,400,4,2.828629e-07,0.447647,0.338340
...,...,...,...,...,...,...,...,...
2716,5,26866401,26867200,800,5,0.000000e+00,0.524355,0.443220
2717,5,26885001,26885700,700,6,0.000000e+00,0.657076,0.574918
2718,5,26885901,26886400,500,5,0.000000e+00,0.517858,0.423792
2719,5,26886601,26886900,300,2,3.711941e-03,0.419719,0.360778


In [15]:
chunks.to_pickle("./data/sig_region.pkl")

In [14]:
def export_to_bed(df, path):
    """
    Export chunk dataframe to BED format.
    Assumes df has 'chr', 'start', 'end' in 1-based inclusive coords.
    """
    bed = df.copy()
    bed['chrom'] = bed['chr'].astype(str)
    bed['chromStart'] = bed['start'] - 1   # convert to 0-based
    bed['chromEnd'] = bed['end']           # end is already exclusive in BED
    cols = ['chrom', 'chromStart', 'chromEnd']
    
    # add any extra annotation columns if you want (e.g., length, n_windows)
    if 'length_bp' in df.columns:
        cols.append('length_bp')
    if 'n_windows' in df.columns:
        cols.append('n_windows')
    
    bed[cols].to_csv(path, sep='\t', header=False, index=False)

# Usage:
export_to_bed(chunks, "dmw_chunks.bed")
